In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
 
sns.set_theme(style='whitegrid')
 
# Create artifacts directory to save our scaler and encoder
os.makedirs('../artifacts', exist_ok=True)
 
print("✅ Imports done")

✅ Imports done


In [2]:
print("Loading full dataset — this may take a minute...")
df = pd.read_csv('../dataset/paysim_data.csv')
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Loading full dataset — this may take a minute...
✅ Loaded: 6,362,620 rows × 11 columns
   Memory: 740.5 MB


In [3]:
import sys
print(sys.executable)
# Should show something like:
# C:\Users\rohan\OneDrive\Desktop\Fin_Gurdain\myenv\Scripts\python.exe
# If it shows a path WITHOUT 'myenv' in it → wrong kernel, switch it

c:\Users\rohan\OneDrive\Desktop\Fin_Gurdain\myenv\Scripts\python.exe


In [4]:
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
df = df.reset_index(drop=True)
 
print(f"After filtering: {len(df):,} rows")
print(f"Fraud count    : {df['isFraud'].sum():,}")
print(f"Fraud rate     : {df['isFraud'].mean()*100:.4f}%")

After filtering: 2,770,409 rows
Fraud count    : 8,213
Fraud rate     : 0.2965%


In [5]:
df['hour'] = df['step'] % 24       # hour of day: 0-23
df['day']  = df['step'] // 24      # day number: 0-30
 
# Cyclical encoding — the key insight from Notebook 02
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_sin']  = np.sin(2 * np.pi * df['day']  / 30)
df['day_cos']  = np.cos(2 * np.pi * df['day']  / 30)
 
print("Temporal features created:")
print(df[['step','hour','hour_sin','hour_cos']].head(3).round(4))

Temporal features created:
   step  hour  hour_sin  hour_cos
0     1     1    0.2588    0.9659
1     1     1    0.2588    0.9659
2     1     1    0.2588    0.9659


In [6]:
df['amount_log'] = np.log1p(df['amount'])

df['amount_to_balance_ratio'] = np.where(
    df['oldbalanceOrg'] > 0,                   # if balance > 0
    df['amount'] / df['oldbalanceOrg'],         # compute ratio
    1.0                                         # if balance = 0, ratio = 1.0 (all-in)
)

print("Amount features created:")
print(df[['amount','amount_log','amount_to_balance_ratio']].describe().round(4))

Amount features created:
             amount    amount_log  amount_to_balance_ratio
count  2.770409e+06  2.770409e+06             2.770409e+06
mean   3.175361e+05  1.192780e+01             1.460154e+02
std    8.877897e+05  1.232800e+00             4.960523e+03
min    0.000000e+00  0.000000e+00             0.000000e+00
25%    8.297354e+04  1.132630e+01             1.000000e+00
50%    1.712609e+05  1.205090e+01             1.000000e+00
75%    3.067912e+05  1.263390e+01             6.494300e+00
max    9.244552e+07  1.834210e+01             3.925476e+06


In [7]:
df['balance_error_orig'] = np.abs(
    df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
)
# Destination account balance error
df['balance_error_dest'] = np.abs(
    df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
)
 
# Log transform (errors also have extreme skew)
df['balance_error_orig_log'] = np.log1p(df['balance_error_orig'])
df['balance_error_dest_log'] = np.log1p(df['balance_error_dest'])
 
# Was the sender's balance completely drained to zero?
df['sender_zeroed'] = (
    (df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0)
).astype(int)
 
# Did the destination account already have zero balance? (mule accounts often do)
df['dest_was_empty'] = (df['oldbalanceDest'] == 0).astype(int)
 
print("Balance anomaly features:")
fraud_means = df[df['isFraud']==1][
    ['balance_error_orig','balance_error_dest','sender_zeroed','dest_was_empty']
].mean().round(4)
legit_means = df[df['isFraud']==0][
    ['balance_error_orig','balance_error_dest','sender_zeroed','dest_was_empty']
].mean().round(4)
 
comparison = pd.DataFrame({'Fraud_mean': fraud_means, 'Legit_mean': legit_means})
comparison['Ratio (fraud/legit)'] = (comparison['Fraud_mean'] /
                                      comparison['Legit_mean'].replace(0, np.nan)).round(1)
print(comparison)

Balance anomaly features:
                     Fraud_mean   Legit_mean  Ratio (fraud/legit)
balance_error_orig   10692.3253  286803.5100                  0.0
balance_error_dest  745138.5856   44302.6608                 16.8
sender_zeroed            0.9755       0.4272                  2.3
dest_was_empty           0.6515       0.1390                  4.7


In [8]:
df['type_encoded'] = (df['type'] == 'TRANSFER').astype(int)
 
print("Type encoding:")
print(df.groupby(['type','type_encoded'])['isFraud'].agg(['count','sum','mean']).round(4))

Type encoding:
                         count   sum    mean
type     type_encoded                       
CASH_OUT 0             2237500  4116  0.0018
TRANSFER 1              532909  4097  0.0077


In [9]:
print("Computing account history features (this takes a minute on full data)...")
 
df = df.sort_values('step').reset_index(drop=True)
 
# How many transactions has this sender made so far? (expanding count)
df['sender_tx_count'] = df.groupby('nameOrig').cumcount()
 
# Has this recipient been used before? (is it a new/unknown recipient?)
df['recipient_tx_count'] = df.groupby('nameDest').cumcount()
df['is_new_recipient']   = (df['recipient_tx_count'] == 0).astype(int)
 
print("✅ Account history features created")
print(f"New recipient rate overall : {df['is_new_recipient'].mean()*100:.1f}%")
print(f"New recipient rate in fraud: {df[df['isFraud']==1]['is_new_recipient'].mean()*100:.1f}%")
print("\\nFraud is more likely when recipient is new → this is a strong feature ✅")

Computing account history features (this takes a minute on full data)...
✅ Account history features created
New recipient rate overall : 18.4%
New recipient rate in fraud: 67.1%
\nFraud is more likely when recipient is new → this is a strong feature ✅


In [10]:
df['has_location'] = 0   # PaySim has no GPS — in production this is real
df['has_device_id'] = 0  # same — production will fill this from the API payload
 
print("has_location and has_device_id columns created")
print("In production: has_location=0 at 11pm = Midnight Ghost signal")
print("Columns in df so far:", [c for c in df.columns if c not in 
      ['step','type','nameOrig','nameDest','isFlaggedFraud']])

has_location and has_device_id columns created
In production: has_location=0 at 11pm = Midnight Ghost signal
Columns in df so far: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'hour', 'day', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'amount_log', 'amount_to_balance_ratio', 'balance_error_orig', 'balance_error_dest', 'balance_error_orig_log', 'balance_error_dest_log', 'sender_zeroed', 'dest_was_empty', 'type_encoded', 'sender_tx_count', 'recipient_tx_count', 'is_new_recipient', 'has_location', 'has_device_id']


In [11]:
# ── Cell 10: Final Feature Set ────────────────────────────────────────────
# These are ALL features going into XGBoost and the Autoencoder.
# This list must exactly match the Redis feature store and Pydantic schema.

FEATURES = [
    # Time — cyclical encoding
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',

    # Amount
    'amount_log', 'amount_to_balance_ratio',

    # Balance anomaly — strongest signals
    'balance_error_orig_log', 'balance_error_dest_log',

    # Binary flags
    'sender_zeroed', 'dest_was_empty', 'is_new_recipient',
    'has_location', 'has_device_id',

    # Account history
    'sender_tx_count', 'recipient_tx_count',

    # Transaction type
    'type_encoded',
]

TARGET = 'isFraud'

print(f"Total features : {len(FEATURES)}")
print("Feature list:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")

# Verify all features exist in df
missing_cols = [f for f in FEATURES if f not in df.columns]
if not missing_cols:
    print("\n✅ All features present in dataframe")
else:
    print(f"\n❌ Missing columns: {missing_cols}")

# Verify no nulls
null_counts = df[FEATURES].isnull().sum()
if null_counts.sum() == 0:
    print("✅ No nulls in any feature — clean dataset ready for training")
else:
    print("⚠️  Nulls found:")
    print(null_counts[null_counts > 0])

Total features : 16
Feature list:
   1. hour_sin
   2. hour_cos
   3. day_sin
   4. day_cos
   5. amount_log
   6. amount_to_balance_ratio
   7. balance_error_orig_log
   8. balance_error_dest_log
   9. sender_zeroed
  10. dest_was_empty
  11. is_new_recipient
  12. has_location
  13. has_device_id
  14. sender_tx_count
  15. recipient_tx_count
  16. type_encoded

✅ All features present in dataframe
✅ No nulls in any feature — clean dataset ready for training


In [12]:
df_sorted = df.sort_values('step').reset_index(drop=True)
 
n         = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
 
df_train = df_sorted.iloc[:train_end]
df_val   = df_sorted.iloc[train_end:val_end]
df_test  = df_sorted.iloc[val_end:]
 
X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_val   = df_val[FEATURES]
y_val   = df_val[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]
 
print(f"Train: {len(X_train):>8,} rows | Fraud: {y_train.sum():>5,} ({y_train.mean()*100:.3f}%)")
print(f"Val  : {len(X_val):>8,} rows | Fraud: {y_val.sum():>5,} ({y_val.mean()*100:.3f}%)")
print(f"Test : {len(X_test):>8,} rows | Fraud: {y_test.sum():>5,} ({y_test.mean()*100:.3f}%)")
print("\\n✅ Time-based split complete — no data leakage")

Train: 1,939,286 rows | Fraud: 3,633 (0.187%)
Val  :  415,561 rows | Fraud:   560 (0.135%)
Test :  415,562 rows | Fraud: 4,020 (0.967%)
\n✅ Time-based split complete — no data leakage


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_val_scaled   = scaler.transform(X_val)         # transform only (no fit)
X_test_scaled  = scaler.transform(X_test)        # transform only (no fit)
 
# Save scaler — we'll need the EXACT same scaler in production
# so that new transactions are scaled identically to training data
scaler_path = '../artifacts/feature_scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
 
print(f"✅ Scaler fitted and saved to {scaler_path}")
mean_val = scaler.mean_
scale_val = scaler.scale_
if mean_val is not None and scale_val is not None:
    print(f"   Scaler mean (first 3 features): {mean_val[:3].round(4)}")
    print(f"   Scaler std  (first 3 features): {scale_val[:3].round(4)}")

✅ Scaler fitted and saved to ../artifacts/feature_scaler.pkl
   Scaler mean (first 3 features): [-0.4433 -0.3571  0.683 ]
   Scaler std  (first 3 features): [0.5591 0.6028 0.353 ]


In [14]:
# ── Cell 13: Save Processed Datasets ─────────────────────────────────────
# Saving as CSV instead of parquet (avoids pyarrow version conflicts)

import json
import os

os.makedirs('../artifacts', exist_ok=True)

cols_to_save = FEATURES + [TARGET, 'step', 'nameOrig', 'nameDest']

df_train[cols_to_save].to_csv('../artifacts/train.csv', index=False)
df_val[cols_to_save].to_csv('../artifacts/val.csv',     index=False)
df_test[cols_to_save].to_csv('../artifacts/test.csv',   index=False)

# Save feature list
with open('../artifacts/feature_list.json', 'w') as f:
    json.dump({'features': FEATURES, 'target': TARGET}, f, indent=2)

print("✅ Saved:")
print(f"   train.csv  — {len(df_train):,} rows")
print(f"   val.csv    — {len(df_val):,} rows")
print(f"   test.csv   — {len(df_test):,} rows")
print(f"   feature_list.json")

✅ Saved:
   train.csv  — 1,939,286 rows
   val.csv    — 415,561 rows
   test.csv   — 415,562 rows
   feature_list.json


In [15]:
# ── Cell 14: Verify ───────────────────────────────────────────────────────
df_check = pd.read_csv('../artifacts/train.csv')

print(f"Shape   : {df_check.shape}")
print(f"Fraud % : {df_check['isFraud'].mean()*100:.4f}%")
print(f"Columns : {list(df_check.columns)}")
print("""
╔══════════════════════════════════════════════════════════════╗
║         NOTEBOOK 03 — COMPLETE ✅                            ║
╠══════════════════════════════════════════════════════════════╣
║  Features engineered : 16                                    ║
║  Train / Val / Test  : 70% / 15% / 15% (time-based)         ║
║  Scaler saved        : artifacts/feature_scaler.pkl          ║
║  Data saved          : artifacts/train/val/test.csv          ║
║  Feature list saved  : artifacts/feature_list.json           ║
╠══════════════════════════════════════════════════════════════╣
║  Next: Notebook 04 — XGBoost Training + ONNX Export         ║
╚══════════════════════════════════════════════════════════════╝
""")

Shape   : (1939286, 20)
Fraud % : 0.1873%
Columns : ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'amount_log', 'amount_to_balance_ratio', 'balance_error_orig_log', 'balance_error_dest_log', 'sender_zeroed', 'dest_was_empty', 'is_new_recipient', 'has_location', 'has_device_id', 'sender_tx_count', 'recipient_tx_count', 'type_encoded', 'isFraud', 'step', 'nameOrig', 'nameDest']

╔══════════════════════════════════════════════════════════════╗
║         NOTEBOOK 03 — COMPLETE ✅                            ║
╠══════════════════════════════════════════════════════════════╣
║  Features engineered : 16                                    ║
║  Train / Val / Test  : 70% / 15% / 15% (time-based)         ║
║  Scaler saved        : artifacts/feature_scaler.pkl          ║
║  Data saved          : artifacts/train/val/test.csv          ║
║  Feature list saved  : artifacts/feature_list.json           ║
╠══════════════════════════════════════════════════════════════╣
║  Next: Notebook 04 — XGBoost Train

In [16]:
import pandas as pd
import os

print("Files in artifacts/:")
for f in os.listdir('../artifacts'):
    size = os.path.getsize(f'../artifacts/{f}') / 1e6
    print(f"  {f:35s} {size:.1f} MB")

df = pd.read_csv('../artifacts/train.csv')
print(f"\nTrain shape: {df.shape}")
print(f"Fraud count: {df['isFraud'].sum()}")

Files in artifacts/:
  autoencoder_best.pt                 0.0 MB
  autoencoder_full.pt                 0.0 MB
  autoencoder_metadata.json           0.0 MB
  feature_list.json                   0.0 MB
  feature_scaler.pkl                  0.0 MB
  model_metadata.json                 0.0 MB
  shap_beeswarm.png                   0.1 MB
  shap_explainer.pkl                  0.2 MB
  shap_feature_importance.png         0.1 MB
  shap_utils.py                       0.0 MB
  shap_waterfall_fraud.png            0.1 MB
  test.csv                            74.8 MB
  train.csv                           345.4 MB
  val.csv                             74.4 MB
  xgboost_fraud.json                  0.1 MB
  xgboost_fraud.onnx                  0.1 MB

Train shape: (1939286, 20)
Fraud count: 3633
